In [ ]:
from neo4j import GraphDatabase, RoutingControl
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt


In [ ]:
URI = "neo4j://localhost:7687"
AUTH=('neo4j', "M@userMaria")

In [ ]:
keywords = ["Urban", "Indoor", "Electric"]
met_values = ["4", "5"]

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    records, _, _ = driver.execute_query(
        '''
        MATCH (t:Type {name: "Bicycling"})<-[:HAS_TYPE]-(s:SAC)
              -[:HAS_MET]->(m:MET),
              (s)-[:HAS_KEYWORD]->(k:Keyword)
        WHERE k.name IN $keywords
          AND m.name IN $met_values
        RETURN t, s, m, k
        ''',
        parameters_={"keywords": keywords, "met_values": met_values},
        database_="neo4j",
        routing_=RoutingControl.READ
    )

records


In [ ]:
cpa = pd.read_csv('cpa.csv')

for record in records:
    code = record['s']['code']
    match = cpa[cpa['SAC'] == int(code)]
    print(match.iloc[0]['SAC'], ", MET:", match.iloc[0]['MET'], ", Activity: ", match.iloc[0][' Description'])


In [ ]:
G = nx.DiGraph()

for record in records:
    t = record["t"]
    s = record["s"]
    m = record["m"]
    k = record["k"]

    G.add_node(t.element_id, label=t["name"], type="Type")
    G.add_node(s.element_id, label=s["code"], type="SAC")
    G.add_node(m.element_id, label=m["name"], type="MET")
    G.add_node(k.element_id, label=k["name"], type="Keyword")

    G.add_edge(s.element_id, t.element_id, label="HAS_TYPE")
    G.add_edge(s.element_id, m.element_id, label="HAS_MET")
    G.add_edge(s.element_id, k.element_id, label="HAS_KEYWORD")


pos = nx.spring_layout(G, seed=42)  

labels = nx.get_node_attributes(G, 'label')
edge_labels = nx.get_edge_attributes(G, 'label')

plt.figure(figsize=(10, 7))
nx.draw(G, pos, labels=labels, with_labels=True, node_size=2000, node_color='lightblue', font_size=10)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
plt.title("Neo4j Network Visualization")
plt.show()
